# Demo 1 — MCP

**Scenario:** ask an LLM about a stock. The LLM has two tool servers.

```
   you (this notebook)
        │
        ▼
   Claude (LLM)  ◄── MCP client (mcp_helpers.py)
        │
        ├──► stock_mcp_server.py    (subprocess via stdio)
        │      get_quote, get_history, get_company_info, get_news_headlines
        │
        └──► viz_mcp_server.py      (subprocess via stdio)
               line_chart, compare_lines
```

| File | Tools |
|------|-------|
| [`../shared/stock_mcp_server.py`](../shared/stock_mcp_server.py) | `get_quote`, `get_history`, `get_company_info`, `get_news_headlines` |
| [`viz_mcp_server.py`](viz_mcp_server.py) | `line_chart`, `compare_lines` |

Data backend: `yfinance` (no API key required). For production, swap to a company-maintained MCP server such as [Alpha Vantage's official one](https://mcp.alphavantage.co/).

## First-time setup

Run once in a terminal:

```bash
python3.12 -m venv .venv && source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

Set the LLM credentials in `.env` (project root). Two options:
- **Anthropic direct:** `ANTHROPIC_API_KEY=sk-ant-...`, `ANTHROPIC_MODEL=claude-sonnet-4-6`
- **OpenRouter (cheaper):** `ANTHROPIC_BASE_URL=https://openrouter.ai/api`, `ANTHROPIC_API_KEY=sk-or-v1-...`, `ANTHROPIC_MODEL=anthropic/claude-sonnet-4.5`

In [ ]:
# ── Bootstrap: resolve paths to shared/ and sibling helpers ──
import sys
from pathlib import Path
HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))   # so `from shared.X import Y` works
sys.path.insert(0, str(HERE))   # so sibling helpers import directly
STOCK_MCP_SERVER = str(ROOT / "shared" / "stock_mcp_server.py")
VIZ_MCP_SERVER = str(HERE / "viz_mcp_server.py")


In [ ]:
%pip install -q mcp anthropic yfinance matplotlib python-dotenv

import os
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

if not os.getenv("ANTHROPIC_API_KEY"):
    print("Note: ANTHROPIC_API_KEY not set in .env — the final 'hand to Claude' cell will skip.")
else:
    print(f"ready  (model={os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')})")

## List tools

In [ ]:
from mcp_helpers import list_tools

await list_tools(STOCK_MCP_SERVER)

In [ ]:
await list_tools(VIZ_MCP_SERVER)

## Call one tool directly

In [ ]:
from mcp_helpers import call_tool

await call_tool(STOCK_MCP_SERVER, "get_quote", ticker="NVDA")

In [ ]:
await call_tool(STOCK_MCP_SERVER, "get_history", ticker="NVDA", days=10)

## Hand both servers to Claude

The LLM gets all six tools. We ask one natural-language question; Claude decides which tools to call, in what order.

In [ ]:
from mcp_helpers import chat_with_claude

await chat_with_claude(
    message=(
        "How has NVDA traded over the last 10 days? "
        "Plot the closing prices and tell me what you see."
    ),
    servers=[STOCK_MCP_SERVER, VIZ_MCP_SERVER],
)